# 03 - Servidor MCP Customizado
> Criando seu proprio servidor MCP em Python

**Modulo:** `05_mcp` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
TEMPLATE = '''
# meu_servidor_mcp.py
import asyncio, json, sqlite3
from mcp.server import Server
from mcp.server.models import InitializationOptions
import mcp.server.stdio, mcp.types as types

app = Server("tarefas-mcp")
conn = sqlite3.connect("tarefas.db")
conn.execute("CREATE TABLE IF NOT EXISTS t (id INTEGER PRIMARY KEY, titulo TEXT, feito BOOLEAN)")
conn.commit()

@app.list_tools()
async def listar():
    return [
        types.Tool(name="criar", description="Cria tarefa.",
            inputSchema={"type":"object","properties":{"titulo":{"type":"string"}},"required":["titulo"]}),
        types.Tool(name="listar", description="Lista tarefas.",
            inputSchema={"type":"object","properties":{}}),
    ]

@app.call_tool()
async def executar(name, arguments):
    if name=="criar":
        cur=conn.execute("INSERT INTO t (titulo,feito) VALUES (?,0)",(arguments["titulo"],))
        conn.commit(); texto=f"Tarefa {cur.lastrowid} criada."
    elif name=="listar":
        rows=conn.execute("SELECT id,titulo,feito FROM t").fetchall()
        texto=json.dumps([{"id":r[0],"titulo":r[1],"feito":bool(r[2])} for r in rows])
    return [types.TextContent(type="text", text=texto)]

async def main():
    async with mcp.server.stdio.stdio_server() as (r,w):
        await app.run(r,w,InitializationOptions(server_name="tarefas",server_version="1.0"))

if __name__=="__main__": asyncio.run(main())
'''

with open('meu_servidor_mcp.py','w',encoding='utf-8') as f: f.write(TEMPLATE)
print('Servidor salvo em meu_servidor_mcp.py')
print('Execute com: python meu_servidor_mcp.py')

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
